# NB_hospital_daily_summary

Builds `caresyncadb.gold.hospital_daily_summary` per the STTM (Silver to Gold v1.0). Grain: one row per hospital per calendar day. Driving table `silver.lab_results` grouped by hospital_id and test_date, LEFT JOIN to hospitals and calendar (joined on `date_key_int`, the silver calendar's yyyyMMdd key), with a LAG() day-over-day trend. Load: full rebuild.

In [0]:
%sql
-- Full rebuild: one row per hospital per day (driving table: silver.lab_results)
CREATE OR REPLACE TABLE caresyncadb.gold.hospital_daily_summary AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                   AS summary_date,
    COUNT(*)                                    AS total_tests,
    COUNT(DISTINCT patient_id)                  AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                   AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')        AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')        AS critical_test_count
  FROM caresyncadb.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                             AS hospital_id,
  h.hospital_name,
  w.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(w.total_tests AS INT)                                             AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                  AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                     AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                     AS critical_test_count,
  CAST(ROUND(w.abnormal_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(w.critical_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                    AS prev_day_total_tests,
  CASE
    WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
    ELSE CAST(ROUND((w.total_tests - w.prev_day_total_tests) * 100.0 / w.prev_day_total_tests, 2) AS DECIMAL(6,2))
  END                                                                    AS test_volume_change_pct
FROM with_lag w
LEFT JOIN caresyncadb.silver.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN caresyncadb.silver.calendar c
  ON c.date_key_int = DATE_FORMAT(w.summary_date, 'yyyyMMdd')

num_affected_rows,num_inserted_rows


In [0]:
# Read the rebuilt gold table and return its record count to the caller (e.g. a job or dbutils.notebook.run)
updated_df = spark.table("caresyncadb.gold.hospital_daily_summary")
record_count = updated_df.count()
print(f"caresyncadb.gold.hospital_daily_summary record count: {record_count}")
dbutils.notebook.exit(str(record_count))